In [1]:
import numpy as np
from keras.models import Sequential
from keras.layers import GlobalAveragePooling2D,Conv2D,MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator    
from keras.applications import MobileNetV2
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score

In [2]:
train="C:\\Users\\adity\\Downloads\\archive (3)\\train"
test="C:\\Users\\adity\\Downloads\\archive (3)\\test"


In [3]:
emotions=['angry', 'happy', 'sad', 'surprise', 'neutral']
import tensorflow as tf


size=(128,128)


train_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)


train_data = train_gen.flow_from_directory(
    train,
    target_size=size,
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    classes=emotions
)

# Validation data (20%)
val_data = train_gen.flow_from_directory(
    train,
    target_size=size,
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    classes=emotions
)

# Test data (separate unseen dataset)
test_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

test_data = test_gen.flow_from_directory(
    test,
    target_size=size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False,
    classes=emotions
)

Found 19341 images belonging to 5 classes.
Found 4835 images belonging to 5 classes.
Found 6043 images belonging to 5 classes.


In [4]:
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
inputs = Input(shape=(128, 128, 3))
base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inputs)

x = base.output
x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D(pool_size=(2,2))(x)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(5, activation='softmax')(x)

model = Model(inputs, outputs)

model.summary()

C:\Users\adity\AppData\Local\Temp\ipykernel_20520\3791187237.py:4: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inputs)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 64, 64,    │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 64, 64,    │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 64, 64,    │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 64, 64,    │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 64, 64,    │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 64, 64,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 65, 65,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 32, 32,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 32, 32,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 32, 32,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 32, 32,    │      2,304 │ block_1_depthwis

 Total params: 3,087,109 (11.78 MB)

 Trainable params: 3,052,613 (11.64 MB)

 Non-trainable params: 34,496 (134.75 KB)

In [6]:
from keras.callbacks import ReduceLROnPlateau
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
model.fit(
    train_data,
    epochs=20,
    validation_data=val_data,
    callbacks=[early_stopping, reduce_lr]
)

c:\Users\adity\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 692s 1s/step - accuracy: 0.5160 - loss: 1.2977 - val_accuracy: 0.3127 - val_loss: 2.0169 - learning_rate: 0.0010
Epoch 2/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 1634s 3s/step - accuracy: 0.6202 - loss: 1.1166 - val_accuracy: 0.5384 - val_loss: 1.4210 - learning_rate: 0.0010
Epoch 3/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 449s 742ms/step - accuracy: 0.6477 - loss: 1.0756 - val_accuracy: 0.4914 - val_loss: 1.5276 - learning_rate: 0.0010
Epoch 4/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 404s 668ms/step - accuracy: 0.6594 - loss: 1.0484 - val_accuracy: 0.3795 - val_loss: 1.9351 - learning_rate: 0.0010
Epoch 5/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 468s 774ms/step - accuracy: 0.7019 - loss: 0.9650 - val_accuracy: 0.6885 - val_loss: 1.0109 - learning_rate: 2.0000e-04
Epoch 6/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 936s 2s/step - accuracy: 0.7203 - loss: 0.9312 - val_accuracy: 0.7022 - val_loss: 0.9802 - learning_rate: 2.0000e-04
Epoch 7/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 430s 711ms/step - accura

KeyboardInterrupt: 

In [7]:
pred2=model.predict(test_data)

189/189 ━━━━━━━━━━━━━━━━━━━━ 40s 204ms/step


In [8]:
print(accuracy_score(test_data.labels, np.argmax(pred2, axis=1)))

0.7516134370345855


In [9]:
from sklearn.metrics import classification_report 
print(classification_report(test_data.classes,np.argmax(pred2, axis=1)))

              precision    recall  f1-score   support

           0       0.74      0.62      0.67       958
           1       0.89      0.87      0.88      1774
           2       0.66      0.63      0.65      1247
           3       0.85      0.85      0.85       831
           4       0.62      0.73      0.67      1233

    accuracy                           0.75      6043
   macro avg       0.75      0.74      0.74      6043
weighted avg       0.76      0.75      0.75      6043



In [10]:
model.save('mobilenet.keras')